In [2]:
import os
import torch
import numpy as np
import pandas as pd
import glob
from sklearn.neighbors import KDTree


In [3]:
def load_csv_files(root_dir, recursive, ignore_dirs, file_list):
    if file_list is not None:
        files = [os.path.join(root_dir, f) for f in file_list]
    elif recursive:
        # search recursively in subdirectories
        files = sorted(glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True))
    else:
        # search only in the root directory
        files = sorted(glob.glob(os.path.join(root_dir, "*.csv")))
    
    # filter out files in ignored directories
    if ignore_dirs:
        files = [f for f in files if not any(
            ig == os.path.basename(os.path.dirname(f)) for ig in ignore_dirs
        )]
    
    if not files:
        search_type = "recursively" if recursive else "in root directory"
        raise FileNotFoundError(f"No CSVs found {search_type} in {root_dir}")
    
    return files

def clean_dataset(df: pd.DataFrame, path: str) -> pd.DataFrame:
    """
    Clean the dataset by removing rows with NaN values in 'x-avg' or 'y-avg' columns.
    """

    if not {"time-rel-seconds", "x-avg", "y-avg"}.issubset(df.columns):
        raise ValueError(f"{path} must have columns: time-rel-seconds,x-avg,y-avg")

    df = df.sort_values("time-rel-seconds").reset_index(drop=True)

    df_cleaned = df.dropna().reset_index(drop=True)
    return df_cleaned

In [ ]:

def load_one(path: str, lookback: int):
    """Convert a single CSV file to a PyTorch Geometric Data object with spatio-temporal edges."""
    df = pd.read_csv(path)

    df = clean_dataset(df, path)
    df = df.loc[:, ["time-rel-seconds", "x-avg", "y-avg", "pupil-size-left-avg", "pupil-size-right-avg"]]
    X = torch.tensor(df[["time-rel-seconds", "x-avg", "y-avg", "pupil-size-left-avg", "pupil-size-right-avg"]].values, dtype=torch.float32)

    def _delta(feature):
        if isinstance(feature, str):
            return df.loc[1:, feature] - df.loc[:-1, feature]
        else:
            return feature[1:] - feature[:-1]
    def _l2_norm(a, b):
        return np.sqrt(np.square(a) + np.square(b))

    dt = _delta("time-rel-seconds")
    print("dt", dt.shape)
    ddt = _delta(dt)
    print("ddt", ddt.shape)

    dx = _delta("x-avg")
    dy = _delta("y-avg")
    d_gaze = np.sqrt(np.square(dx) + np.square(dy))
    v_gaze = d_gaze / dt
    d_v_gaze = _delta(v_gaze)
    a_gaze = d_v_gaze / ddt

    d_pupil_left = _delta("pupil-size-left-avg")
    d_pupil_right = _delta("pupil-size-right-avg")
    d_pupil = _l2_norm(d_pupil_left, d_pupil_right)
    v_pupil = d_pupil / dt
    d_v_pupil = _delta(v_pupil)
    a_pupil = d_v_pupil / ddt

    # edges_temporal_weights = torch.tensor([dt, v_gaze, v_pupil, a_gaze, a_pupil], dtype=torch.float32).t()  # [num_edges, 5]
    edges_temporal = []    # connect each node to previous lookback steps


    n = len(df)
    # k = getattr("k", 10)
    # reserve integer index space [n, k], use -1 as sentinel for "no neighbor"
    print("n", n, "k", k)
    edges_spatial = torch.full((n, k), -1, dtype=torch.long)
    for i in range(n):
        # Look back up to lookback steps (or fewer if near start)
        lb = min(lookback, i)
        for j in range(1, lb + 1):
            edges_temporal.append((i - j, i))  # previous node -> current node
        
        # spatial neighbors based on KDTree
        tree = KDTree(X[max(0, i):i+1, 1:3].numpy()) # IF THE DATASET (i) IS TOO SMALL, THIS WILL BREAK
        s = X[i, 1:3].numpy().reshape(1, -1)
        print(s)
        dist, ind = tree.query(s, k=k+1)  # +1 to exclude self
        neighbors = ind[0][1:] + max(0, i)  # adjust indices
        print(neighbors)
        edges_spatial[i] = torch.tensor(neighbors, dtype=torch.long)

    print("spatial", edges_spatial)
    print("temporal", edges_temporal)




In [20]:
lookback = 3
k = 5
root_dir = "."
recursive = True
ignore_dirs = []
file_list = None

"""
Load all CSV files from directory and convert to graphs.
If file_list is provided, search for the files in the list in root_dir.
"""
k = k  # number of spatial neighbors
lookback = lookback

files = load_csv_files(root_dir, recursive, ignore_dirs, file_list)

# pre-load all graphs into memory for simplicity
# graphs = [load_one(p, lookback) for p in files]
# print(f"Loaded {len(graphs)} graphs from {root_dir}")
print(files[0])
load_one(files[0], lookback)

./data/processed/cog-load-mini/s_001.csv
dt (94026,)
ddt (94026,)
n 94027 k 5
[[978. 600.]]


ValueError: k must be less than or equal to the number of training points